# ONG v3 — Tahap A Bake-off (L4)

Train + evaluate candidate backbones on the identical protocol, then auto-build a
paste-ready results block for `ONGv3_progress.Rmd`.

> **⚠️ UPDATE 2026-06-22 — read first.** The first bake-off undershot on genus because of
> (1) checkpoint selection by val *macro* discarded the high-*global* checkpoints (DINOv2
> hit val_top1 0.72 but we saved the macro-peak epoch at 0.526), (2) double-balancing
> (WeightedRandomSampler + CB-Focal) suppressed the true biological prior — Bulbophyllum &
> Dendrobium really ARE the two largest orchid genera, so balancing fights reality and
> collapsed them (Dendrobium 97%→46%/23%), (3) early overfit. **Fix applied:**
> `03_train_bakeoff_colab.py` now ALWAYS saves BOTH `best_model_macro.pth` +
> `best_model_global.pth`, and `run_bakeoff_all_colab.py` evaluates BOTH. **Product-first
> path = run the PILOT (Sel 3.6):** DINOv2 with the prior-respecting recipe
> `--sampler-power 0 --loss ce`. Run the full 4-model bake-off only for the paper.

**Order (predicted-best / cheapest first):** bioclip2 → dinov2l → convnextv2l → effnetv2l.

**FROZEN baseline to beat** (clean + merged + deterministic split, 16,701 img / 120 genera):
**macro top-1 74.0%** (support≥5: 76.2%), macro top-5 84.8%, global 90.1%,
species R@5 73.4%, genus R@5 94.1%, ECE 0.082. ⚠️ Baseline is OPTIMISTIC (train/test
overlap) — bake-off models get an honest holdout, so even matching it is an improvement.

**Drive layout expected** under `MyDrive/ONG_v3/`:
```
ONG_v3/
  scripts/ 03_train_bakeoff_colab.py  13_evaluate.py  run_bakeoff_all_colab.py
  data/splits/*.csv
  models/{bioclip2,dinov2l,convnextv2l,effnetv2l}/best_model_{macro,global}.pth + vocab.json
  photos.zip            (unzipped to /content/photos in Sel 3)
```

**Product-first (recommended now):** Sel 1 → 2 → 3 → **Sel 3.6 (PILOT)** → Sel 5.
**Full bake-off (paper):** Sel 1 → 2 → 3 → Sel 4 → Sel 5.

## Sel 1 — Setup (mount Drive + install)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install -U timm open_clip_torch faiss-cpu
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU — switch Runtime > Change runtime type > L4.'

## Sel 2 — Define ROOT and verify scripts + splits

In [ ]:
import os
ROOT = '/content/drive/MyDrive/ONG_v3'
need = ['03_train_bakeoff_colab.py', '13_evaluate.py', 'run_bakeoff_all_colab.py']
missing = [f for f in need if not os.path.exists(f'{ROOT}/scripts/{f}')]
print('scripts present:', {f: os.path.exists(f'{ROOT}/scripts/{f}') for f in need})
print('splits present :', os.path.exists(f'{ROOT}/data/splits/train_live.csv'))
assert not missing, f'Upload these to {ROOT}/scripts/: {missing}'
assert os.path.exists(f'{ROOT}/data/splits/train_live.csv'), 'Upload data/splits/*.csv to Drive.'
print

## Sel 3 — Unzip photos to local disk (fast; never train off Drive)

In [ ]:
# Adjust the zip path if yours differs. Result structure: /content/photos/{Genus}/*.jpg
!unzip -q -o "{ROOT}/photos.zip" -d /content/
import os
assert os.path.isdir('/content/photos'), 'Expected /content/photos after unzip — check the zip layout.'
print('genera folders:', len(os.listdir('/content/photos')))
print('sample:', sorted(os.listdir('/content/photos'))[:8])

In [ ]:
# ============================================================================
# Sel 3.6 — PILOT (corrected, prior-respecting recipe)  ← PRODUCT-FIRST PATH
# DINOv2 with NO double-balancing: --sampler-power 0 --loss ce (v2-style, proven 71%).
# Saves BOTH best_model_macro.pth + best_model_global.pth, then evaluates BOTH on the
# frozen test split. ~15-20 CU on L4. Run this INSTEAD of Sel 4 for the product path.
# (Skip Sel 3.5 smoke test; this pilot exercises the whole pipeline anyway.)
# ============================================================================
M = 'dinov2l'
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model {M} --drive-root {ROOT} --photos-root /content/photos --sampler-power 0 --loss ce

# Evaluate BOTH selection checkpoints (macro → eval/<M>, global → eval/<M>_global):
for _tag in ['macro', 'global']:
    _ck  = f'{ROOT}/models/{M}/best_model_{_tag}.pth'
    _out = f'{ROOT}/eval/{M}' if _tag == 'macro' else f'{ROOT}/eval/{M}_{_tag}'
    print(f'\n{"="*70}\n[EVAL] {M} [{_tag}-selected checkpoint]\n{"="*70}')
    get_ipython().system(
        f'python {ROOT}/scripts/13_evaluate.py --model vit_large_patch14_reg4_dinov2.lvd142m '
        f'--img-size 448 --checkpoint {_ck} --vocab {ROOT}/models/{M}/vocab.json '
        f'--splits-dir {ROOT}/data/splits --photos-root /content/photos --out-dir {_out}'
    )
print('\nPilot done. Run Sel 5 to build the comparison table (vs baseline macro 74.0 / global 90.1).')

## Sel 3.5 — Smoke test (run ONCE before the multi-hour bake-off)

Trains the cheapest model (bioclip2 @224) for 1 warm-up + 1 fine-tune epoch, then
evaluates on 256 images. ~10-15 min, a fraction of a compute unit. It exercises the
WHOLE pipeline end-to-end — data loading, the backbone-unfreeze step (where OOM would
show up), checkpoint save, and the eval loader for the openclip checkpoint — so a clean
table here means Sel 4 is safe to leave running. Output goes to `eval/_smoke`; note the
real Sel 4 run retrains bioclip2 from scratch and overwrites `models/bioclip2/`.

In [ ]:
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model bioclip2 --drive-root {ROOT} --photos-root /content/photos --warmup-epochs 1 --finetune-epochs 1 --early-stop 99
!python {ROOT}/scripts/13_evaluate.py --arch openclip --model hf-hub:imageomics/bioclip-2 --img-size 224 --checkpoint {ROOT}/models/bioclip2/best_model.pth --vocab {ROOT}/models/bioclip2/vocab.json --splits-dir {ROOT}/data/splits --photos-root /content/photos --out-dir {ROOT}/eval/_smoke --limit 256

## Sel 4 — Evaluate the four trained models (eval-only)

The cell below uses `--skip-train`, so it only (re)evaluates models already saved under
`models/<key>/` — that's the current case (all 4 trained). Each writes
`eval/<key>/results.json` on the identical `test_live` split.

- **Need to (re)train one model** (e.g. effnetv2l crashed): drop `--skip-train` and add
  `--models effnetv2l` to train + eval just that one.
- **Full bake-off from scratch:** `%run {ROOT}/scripts/run_bakeoff_all_colab.py --root {ROOT} --photos /content/photos` (several hours on L4; `best_model.pth` saved on every new best, so disconnects are recoverable).

In [ ]:
%run {ROOT}/scripts/run_bakeoff_all_colab.py --root {ROOT} --photos /content/photos --skip-train

## Sel 5 — Build paste-ready results block for `ONGv3_progress.Rmd`

Reads every `eval/<key>/results.json` (including `eval/<key>_global` when present),
prints a dated Markdown changelog entry, and saves it to `eval/bakeoff_summary_<date>.md`
on Drive. Copy the printed block into the Changelog of `ONGv3_progress.Rmd` and the
Tahap A table in `ONGv3_plan.Rmd`.

In [ ]:
import json, datetime
from pathlib import Path

ORDER = ['bioclip2', 'dinov2l', 'convnextv2l', 'effnetv2l']
NAMES = {'bioclip2': 'BioCLIP-2 ViT-L/14', 'dinov2l': 'DINOv2 ViT-L/14',
         'convnextv2l': 'ConvNeXt-V2-L', 'effnetv2l': 'EfficientNetV2-L'}
# FROZEN deterministic baseline (clean+merged split, 16,701 img / 120 genera). OPTIMISTIC
# (train/test overlap) → matching it already counts as a win. macro≥5 = 76.2%.
BASE = dict(macro_top1=74.0, macro_top5=84.8, global_top1=90.1, sp_r5=73.4, gn_r5=94.1)

def label_name(lab):
    if lab.endswith('_global'):
        return NAMES.get(lab[:-7], lab[:-7]) + ' [global-sel]'
    return NAMES.get(lab, lab)

# Include BOTH selection checkpoints when present: eval/<key> (macro) + eval/<key>_global.
labels = []
for k in ORDER:
    labels.append(k)
    if (Path(ROOT) / 'eval' / f'{k}_global').exists():
        labels.append(f'{k}_global')

def row(lab):
    f = Path(ROOT) / 'eval' / lab / 'results.json'
    if not f.exists():
        return None
    r = json.loads(f.read_text())
    c = r.get('classification', {}) or {}
    q = r.get('retrieval', {}) or {}
    pct = lambda d, k: round(d[k] * 100, 1) if d.get(k) is not None else None
    ece = round(c['ece'], 3) if c.get('ece') is not None else None
    return dict(macro_top1=pct(c, 'macro_top1'), macro_top5=pct(c, 'macro_top5'),
                global_top1=pct(c, 'global_top1'), sp_r5=pct(q, 'species_recall@5'),
                gn_r5=pct(q, 'genus_recall@5'), sp_r1=pct(q, 'species_recall@1'), ece=ece)

f2 = lambda x: '—' if x is None else f'{x}'
date = datetime.date.today().isoformat()
L = []
L.append(f'## {date} — Tahap A bake-off results (L4, identical protocol)\n')
L.append('Baseline = FROZEN deterministic effb4 (OPTIMISTIC: train/test overlap; macro≥5 76.2%). '
         '`[global-sel]` rows = `best_model_global.pth` (checkpoint chosen by val global top-1).\n')
L.append('| Model | macro T1 | macro T5 | global T1 | species R@5 | genus R@5 | ECE |')
L.append('|-------|----------|----------|-----------|-------------|-----------|-----|')
L.append(f"| baseline effb4 (FROZEN) | **{BASE['macro_top1']}** | {BASE['macro_top5']} | "
         f"{BASE['global_top1']} | {BASE['sp_r5']} | {BASE['gn_r5']} | 0.082 |")
best_cls = best_ret = None
for lab in labels:
    r = row(lab)
    if r is None:
        continue
    L.append(f"| {label_name(lab)} | **{f2(r['macro_top1'])}** | {f2(r['macro_top5'])} | "
             f"{f2(r['global_top1'])} | {f2(r['sp_r5'])} | {f2(r['gn_r5'])} | {f2(r['ece'])} |")
    if r['macro_top1'] is not None and (best_cls is None or r['macro_top1'] > best_cls[1]):
        best_cls = (lab, r['macro_top1'])
    if r['sp_r5'] is not None and (best_ret is None or r['sp_r5'] > best_ret[1]):
        best_ret = (lab, r['sp_r5'])
L.append('')
if best_cls:
    d = round(best_cls[1] - BASE['macro_top1'], 1)
    L.append(f"- **Winner genus (macro top-1):** {label_name(best_cls[0])} = {best_cls[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
if best_ret:
    d = round(best_ret[1] - BASE['sp_r5'], 1)
    L.append(f"- **Winner retrieval (species R@5):** {label_name(best_ret[0])} = {best_ret[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
L.append('- Checkpoints: `models/<key>/best_model_{macro,global}.pth`; '
         'metrics in `eval/<key>[_global]/results.json`.')
snippet = '\n'.join(L)

out = Path(ROOT) / 'eval' / f'bakeoff_summary_{date}.md'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(snippet, encoding='utf-8')
print(snippet)
print(f'\nSaved -> {out}')
print('Download it (Files panel) and paste the block into ONGv3_progress.Rmd (Changelog)')
print('and the Tahap A table in ONGv3_plan.Rmd.')